# SpyCloud Threat Landscape Analysis
## Organizational Risk Assessment & Executive Reporting

This notebook provides a comprehensive threat landscape analysis using SpyCloud data,
calculating organizational risk scores, trend analysis, and generating executive-ready reports.

**Sections:**
1. Setup & Configuration
2. Exposure Landscape Overview
3. Breach Intelligence Analysis
4. Organizational Risk Score
5. Firewall & Network Posture
6. Remediation Effectiveness
7. Executive Summary Report

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import msticpy as mp
mp.init_notebook(namespace=globals())
from msticpy.data import QueryProvider
qry_prov = QueryProvider('LogAnalytics')
qry_prov.connect(mp.WorkspaceConfig())
LOOKBACK = 90
print(f'Connected. Analysis period: {LOOKBACK} days')

## 2. Exposure Landscape Overview

In [ ]:
# Total exposure statistics
stats_query = f"""
SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| summarize 
    TotalExposures = count(),
    UniqueUsers = dcount(email),
    UniqueDomains = dcount(email_domain),
    CriticalExposures = countif(severity == 25),
    HighExposures = countif(severity == 20),
    MediumExposures = countif(severity == 5),
    LowExposures = countif(severity == 2),
    PlaintextPasswords = countif(password_type == 'plaintext'),
    InfectedDevices = dcount(infected_machine_id),
    TargetDomains = dcount(target_domain)
"""
stats = qry_prov.exec_query(stats_query)
print('=== EXPOSURE LANDSCAPE ===')
for col in stats.columns:
    print(f'  {col}: {stats[col].values[0]:,}')

# Severity trend over time
trend_query = f"""
SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| summarize Count = count() by bin(TimeGenerated, 1d), severity
| order by TimeGenerated asc
"""
trend_df = qry_prov.exec_query(trend_query)
fig = px.area(trend_df, x='TimeGenerated', y='Count', color='severity',
              title=f'Exposure Trend ({LOOKBACK} Days)',
              color_discrete_map={2: '#3498db', 5: '#f39c12', 20: '#e74c3c', 25: '#8e44ad'})
fig.show()

In [ ]:
# Top exposed domains and users
domain_query = f"""
SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| summarize Exposures = count(), Users = dcount(email), MaxSeverity = max(severity) by email_domain
| order by Exposures desc | take 15
"""
domain_df = qry_prov.exec_query(domain_query)
fig = px.bar(domain_df, x='email_domain', y='Exposures', color='MaxSeverity',
             title='Top 15 Exposed Domains', text='Users')
fig.show()

user_query = f"""
SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| summarize Exposures = count(), MaxSev = max(severity), Domains = dcount(target_domain) by email
| order by Exposures desc | take 20
"""
user_df = qry_prov.exec_query(user_query)
fig = px.bar(user_df, x='email', y='Exposures', color='MaxSev',
             title='Top 20 Exposed Users', hover_data=['Domains'])
fig.update_xaxes(tickangle=45)
fig.show()

## 3. Breach Intelligence Analysis

In [ ]:
# Breach catalog analysis
catalog_query = """
SpyCloudBreachCatalog_CL
| summarize BreachCount = count(), TotalRecords = sum(num_records) by breach_category
| order by BreachCount desc
"""
catalog_df = qry_prov.exec_query(catalog_query)
fig = px.pie(catalog_df, values='BreachCount', names='breach_category',
             title='Breach Categories Affecting Organization')
fig.show()

# Malware family distribution
malware_query = """
SpyCloudBreachCatalog_CL
| where isnotempty(malware_family)
| summarize Breaches = count(), Records = sum(num_records) by malware_family
| order by Records desc | take 15
"""
malware_df = qry_prov.exec_query(malware_query)
if not malware_df.empty:
    fig = px.bar(malware_df, x='malware_family', y='Records',
                 title='Top Malware Families', color='Breaches')
    fig.show()

## 4. Organizational Risk Score

In [ ]:
# Calculate composite risk score (0-100)
s = stats.iloc[0]
total = max(s['TotalExposures'], 1)

# Factor 1: Critical exposure ratio (0-25)
critical_ratio = (s['CriticalExposures'] + s['HighExposures']) / total
f1_score = min(25, critical_ratio * 100)

# Factor 2: Plaintext password ratio (0-25)
plaintext_ratio = s['PlaintextPasswords'] / total
f2_score = min(25, plaintext_ratio * 100)

# Factor 3: Infected device count (0-25)
f3_score = min(25, s['InfectedDevices'] * 2.5)

# Factor 4: Remediation gap (0-25)
rem_query = f"""
let exposed = SpyCloudBreachWatchlist_CL | where TimeGenerated > ago(30d) and severity >= 20 | distinct email;
let remediated = SpyCloud_ConditionalAccessLogs_CL | where TimeGenerated > ago(30d) and ActionStatus == 'Success' | distinct Email;
exposed | summarize TotalExposed = count(), Remediated = countif(email in (remediated))
"""
try:
    rem_df = qry_prov.exec_query(rem_query)
    rem_rate = rem_df.iloc[0]['Remediated'] / max(rem_df.iloc[0]['TotalExposed'], 1)
except:
    rem_rate = 0
f4_score = 25 * (1 - rem_rate)

total_risk = f1_score + f2_score + f3_score + f4_score

# Display risk gauge
fig = go.Figure(go.Indicator(
    mode='gauge+number+delta',
    value=total_risk,
    title={'text': 'Organizational Risk Score'},
    gauge={'axis': {'range': [0, 100]},
           'bar': {'color': '#e74c3c' if total_risk > 70 else '#f39c12' if total_risk > 40 else '#2ecc71'},
           'steps': [{'range': [0, 40], 'color': '#d5f5e3'},
                     {'range': [40, 70], 'color': '#fdebd0'},
                     {'range': [70, 100], 'color': '#fadbd8'}]}))
fig.show()

print(f'\nRisk Breakdown:')
print(f'  Critical/High Exposure Ratio: {f1_score:.1f}/25')
print(f'  Plaintext Password Ratio: {f2_score:.1f}/25')
print(f'  Infected Device Impact: {f3_score:.1f}/25')
print(f'  Remediation Gap: {f4_score:.1f}/25')
print(f'  TOTAL: {total_risk:.1f}/100')

## 5. Firewall & Network Posture

In [ ]:
# Check SpyCloud IP overlap with firewall events
fw_overlap_query = f"""
let spycloudIPs = SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago(30d) and severity >= 20
| mv-expand ip = ip_addresses | project IP = tostring(ip);
CommonSecurityLog
| where TimeGenerated > ago(7d)
| summarize AllowCount = countif(DeviceAction has_any ('Allow', 'Accept')),
            DenyCount = countif(DeviceAction has_any ('Deny', 'Drop', 'Block'))
            by DeviceVendor
| where AllowCount > 0 or DenyCount > 0
"""
try:
    fw_df = qry_prov.exec_query(fw_overlap_query)
    if not fw_df.empty:
        fig = px.bar(fw_df, x='DeviceVendor', y=['AllowCount', 'DenyCount'],
                     title='Firewall Events for SpyCloud IPs', barmode='group')
        fig.show()
    else:
        print('No firewall correlation data available.')
except:
    print('CommonSecurityLog not available. Install firewall connectors for correlation.')

## 6. Remediation Effectiveness

In [ ]:
# Remediation metrics
rem_metrics_query = f"""
SpyCloud_ConditionalAccessLogs_CL
| where TimeGenerated > ago({LOOKBACK}d)
| summarize 
    TotalActions = count(),
    Successes = countif(ActionStatus == 'Success'),
    Failures = countif(ActionStatus == 'Failed'),
    PasswordResets = countif(PasswordReset == true),
    SessionRevocations = countif(SessionsRevoked == true),
    GroupAssignments = countif(AddedToCAGroup == true)
"""
try:
    rem_metrics = qry_prov.exec_query(rem_metrics_query)
    print('=== REMEDIATION METRICS ===')
    for col in rem_metrics.columns:
        print(f'  {col}: {rem_metrics[col].values[0]:,}')
except:
    print('No remediation data available yet.')

# Device remediation
mde_metrics_query = f"""
Spycloud_MDE_Logs_CL
| where TimeGenerated > ago({LOOKBACK}d)
| summarize 
    TotalActions = count(),
    Isolations = countif(Action == 'Isolate'),
    Scans = countif(Action == 'Scan'),
    Tags = countif(Action == 'Tag')
"""
try:
    mde_metrics = qry_prov.exec_query(mde_metrics_query)
    print('\n=== DEVICE REMEDIATION ===')
    for col in mde_metrics.columns:
        print(f'  {col}: {mde_metrics[col].values[0]:,}')
except:
    print('No device remediation data available yet.')

## 7. Executive Summary Report

In [ ]:
from datetime import datetime

print('=' * 70)
print('SPYCLOUD THREAT INTELLIGENCE - EXECUTIVE SUMMARY')
print(f'Report Date: {datetime.now().strftime("%Y-%m-%d %H:%M UTC")}')
print(f'Analysis Period: {LOOKBACK} days')
print('=' * 70)

print(f'\nORGANIZATIONAL RISK SCORE: {total_risk:.0f}/100')
if total_risk > 70:
    print('STATUS: CRITICAL - Immediate action required')
elif total_risk > 40:
    print('STATUS: ELEVATED - Active monitoring and remediation needed')
else:
    print('STATUS: MANAGED - Continue monitoring')

print(f'\nKEY METRICS:')
s = stats.iloc[0]
print(f'  Total Exposures: {s["TotalExposures"]:,}')
print(f'  Unique Users Affected: {s["UniqueUsers"]:,}')
print(f'  Critical Exposures (sev 25): {s["CriticalExposures"]:,}')
print(f'  Infostealer Exposures (sev 20): {s["HighExposures"]:,}')
print(f'  Plaintext Passwords: {s["PlaintextPasswords"]:,}')
print(f'  Infected Devices: {s["InfectedDevices"]:,}')
print(f'  Remediation Rate: {rem_rate*100:.1f}%')

print(f'\nTOP RECOMMENDATIONS:')
if s['CriticalExposures'] > 0:
    print(f'  1. [CRITICAL] {s["CriticalExposures"]} session cookie exposures require immediate session revocation')
if s['PlaintextPasswords'] > 0:
    print(f'  2. [HIGH] {s["PlaintextPasswords"]} plaintext passwords require immediate reset')
if s['InfectedDevices'] > 0:
    print(f'  3. [HIGH] {s["InfectedDevices"]} infected devices require isolation and forensic investigation')
if rem_rate < 0.8:
    print(f'  4. [MEDIUM] Remediation rate is {rem_rate*100:.0f}% - target is 80%+')
print(f'  5. [ONGOING] Continue monitoring exposure trends and adjusting severity thresholds')

print(f'\n{"=" * 70}')
print('END OF REPORT')